# SHOTpy Tutorial: Using SHOT from Python

This notebook provides a comprehensive guide to using **SHOTpy**, the Python interface for the **SHOT** (Supporting Hyperplane Optimization Toolkit) solver.

SHOT is a solver for mixed-integer nonlinear programming (MINLP) problems that uses a combination of polyhedral outer approximation, extended supporting hyperplanes, and other techniques.

## Table of Contents
1. [Getting Started](#getting-started)
2. [Creating Variables](#creating-variables)
3. [Linear Problems](#linear-problems)
4. [Mixed-Integer Problems](#mixed-integer-problems)
5. [Quadratic Problems](#quadratic-problems)
6. [Nonlinear Problems](#nonlinear-problems)
7. [Reading Problems from Files](#reading-from-files)
8. [Configuring Solver Settings](#solver-settings)
9. [Advanced Features](#advanced-features)

## 1. Getting Started {#getting-started}

First, import the SHOTpy module and create an environment.

In [ ]:
import sys
import os

# Add current directory to Python path to find SHOTpy module
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())

import SHOTpy

# Create a SHOT solver and get its environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()
print("SHOT solver and environment created successfully!")

## 2. Creating Variables {#creating-variables}

Variables are the fundamental building blocks of optimization problems. SHOTpy supports:
- **Real** (continuous) variables
- **Integer** variables
- **Binary** variables

In [ ]:
# Create a continuous variable: x ∈ [0, 10]
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)

# Create an integer variable: y ∈ {0, 1, 2, ..., 5}
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 5.0)

# Create a binary variable: z ∈ {0, 1}
z = SHOTpy.Variable("z", 2, SHOTpy.VariableType.Binary, 0.0, 1.0)

print(f"Created variables: {x.name}, {y.name}, {z.name}")

## 3. Linear Problems {#linear-problems}

Let's solve a simple linear program (LP):

$$
\begin{align*}
\text{minimize} \quad & 2x + 3y \\
\text{subject to} \quad & x + y \geq 1 \\
& x, y \geq 0
\end{align*}
$$

In [ ]:
# Create solver and get environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()

# Create a new problem
problem = SHOTpy.Problem(env)

# Add variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 100.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 100.0)
problem.addVariable(x)
problem.addVariable(y)

# Create linear objective: minimize 2x + 3y
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(2.0, x))
objective.add(SHOTpy.LinearTerm(3.0, y))
problem.setObjective(objective)

# Add constraint: x + y >= 1
constraint = SHOTpy.LinearConstraint(0, "c1", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Finalize and solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

# Get results
sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

## 4. Mixed-Integer Problems {#mixed-integer-problems}

SHOT excels at solving mixed-integer problems. Here's a simple MIP:

$$
\begin{align*}
\text{maximize} \quad & 3x + 2y \\
\text{subject to} \quad & 2x + y \leq 10 \\
& x, y \in \mathbb{Z}_+
\end{align*}
$$

In [ ]:
# Create a MIP problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

# Add integer variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Integer, 0.0, 100.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 100.0)
problem.addVariable(x)
problem.addVariable(y)

# Objective: maximize 3x + 2y
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Maximize)
objective.add(SHOTpy.LinearTerm(3.0, x))
objective.add(SHOTpy.LinearTerm(2.0, y))
problem.setObjective(objective)

# Constraint: 2x + y <= 10
constraint = SHOTpy.LinearConstraint(0, "budget", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint.add(SHOTpy.LinearTerm(2.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]}, y = {sol.point[1]}")

## 5. Quadratic Problems {#quadratic-problems}

SHOT can handle quadratic objectives and constraints (QCQP):

$$
\begin{align*}
\text{minimize} \quad & x^2 + y^2 \\
\text{subject to} \quad & x + y \geq 2 \\
& x, y \geq 0
\end{align*}
$$

In [ ]:
# Create problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

# Variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Quadratic objective: x^2 + y^2
objective = SHOTpy.QuadraticObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.QuadraticTerm(1.0, x, x))  # x^2
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))  # y^2
problem.setObjective(objective)

# Linear constraint: x + y >= 2
constraint = SHOTpy.LinearConstraint(0, "sum_constraint", 2.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Quadratic Constraints

You can also add quadratic constraints, such as circles or ellipses:

In [ ]:
# Problem: minimize x + y subject to x^2 + y^2 <= 4 (inside a circle)
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, -10.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, -10.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Linear objective
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
objective.add(SHOTpy.LinearTerm(1.0, y))
problem.setObjective(objective)

# Quadratic constraint: x^2 + y^2 <= 4
constraint = SHOTpy.QuadraticConstraint(0, "circle", -SHOTpy.SHOT_DBL_MAX, 4.0)
constraint.add(SHOTpy.QuadraticTerm(1.0, x, x))
constraint.add(SHOTpy.QuadraticTerm(1.0, y, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")
print(f"Distance from origin: {(sol.point[0]**2 + sol.point[1]**2)**0.5:.4f}")

## 6. Nonlinear Problems {#nonlinear-problems}

SHOT is designed for nonlinear problems. You can use various nonlinear expressions:
- Exponential and logarithm: `SHOTpy.exp()`, `SHOTpy.log()`
- Trigonometric: `SHOTpy.sin()`, `SHOTpy.cos()`, `SHOTpy.tan()`
- Square root: `SHOTpy.sqrt()`
- Power functions: `x**n`
- Arithmetic operators: `+`, `-`, `*`, `/`

### Example: Geometric Programming

$$
\begin{align*}
\text{minimize} \quad & e^x + e^y \\
\text{subject to} \quad & e^{x+y} \leq 10 \\
& x, y \geq 0.1
\end{align*}
$$

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Nonlinear objective: exp(x) + exp(y)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.exp(x) + SHOTpy.exp(y))
problem.setObjective(objective)

# Nonlinear constraint: exp(x + y) <= 10
constraint = SHOTpy.NonlinearConstraint(0, "nl_constraint", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint.add(SHOTpy.exp(x + y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Example: Mixed Terms

You can combine linear, quadratic, and nonlinear terms:

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 5.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 5.0)
problem.addVariable(x)
problem.addVariable(y)

# Mixed objective: x + y^2 + log(x)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))           # Linear term
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))     # Quadratic term
objective.add(SHOTpy.log(x))                        # Nonlinear term
problem.setObjective(objective)

# Constraint: x + y >= 1
constraint = SHOTpy.LinearConstraint(0, "bound", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Advanced Nonlinear Expressions

Complex nested expressions are also supported:

In [ ]:
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 3.14)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 1.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Complex expression: exp(x) * sin(x) + log(y) * cos(x)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
expr = SHOTpy.exp(x) * SHOTpy.sin(x) + SHOTpy.log(y) * SHOTpy.cos(x)
objective.add(expr)
problem.setObjective(objective)

# Simple constraint
constraint = SHOTpy.LinearConstraint(0, "bound", 0.5, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
problem.addConstraint(constraint)

problem.finalize()
print("\nProblem structure:")
print(problem.toString())

## 7. Reading Problems from Files {#reading-from-files}

SHOT can read optimization problems from various file formats:
- **OSiL** (Optimization Services Instance Language)
- **GAMS** files (.gms)
- Other formats supported by modeling systems

In [ ]:
# Example: Reading from an OSiL file
# Note: Replace the path with an actual OSiL file path

# problem = SHOTpy.Problem(env)
# problem_file = "/path/to/problem.osil"
# 
# # Read the problem from file
# if problem.importModel(problem_file):
#     print(f"Successfully loaded problem from {problem_file}")
#     problem.finalize()
#     
#     solver = SHOTpy.Solver(env, problem)
#     solver.solveProblem()
#     
#     print(f"Objective value: {solver.getPrimalObjectiveValue():.4f}")
# else:
#     print("Failed to load problem file")

print("See commented code above for file loading example")

## 8. Configuring Solver Settings {#solver-settings}

SHOT provides extensive configuration options through its settings system.

### Accessing and Modifying Settings

In [ ]:
# Create solver and environment first
solver = SHOTpy.Solver()
env = solver.getEnvironment()

# Create a simple problem
problem = SHOTpy.Problem(env)
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)

objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)

problem.finalize()
solver.setProblem(problem)

# Get current values
print("Current Settings:")
time_limit = solver.getDoubleSetting("TimeLimit", "Termination")
print(f"  Time limit: {time_limit} seconds")

abs_gap = solver.getDoubleSetting("ObjectiveGap.Absolute", "Termination")
print(f"  Absolute gap tolerance: {abs_gap}")

# Modify settings
print("\nModifying settings...")
solver.updateSetting("TimeLimit", "Termination", 300.0)  # 5 minutes
solver.updateSetting("ObjectiveGap.Absolute", "Termination", 1e-6)

# Verify changes
new_time = solver.getDoubleSetting("TimeLimit", "Termination")
new_gap = solver.getDoubleSetting("ObjectiveGap.Absolute", "Termination")
print(f"  New time limit: {new_time} seconds")
print(f"  New absolute gap: {new_gap}")

### Common Settings

Here are some frequently used settings:

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)
problem.finalize()

solver.setProblem(problem)

# Termination criteria
solver.updateSetting("TimeLimit", "Termination", 600.0)              # Max time in seconds
solver.updateSetting("ObjectiveGap.Absolute", "Termination", 1e-5)   # Absolute gap
solver.updateSetting("ObjectiveGap.Relative", "Termination", 1e-4)   # Relative gap

# Output settings
solver.updateSetting("Console.LogLevel", "Output", 1)                # 0=off, 1=summary, 2=detailed

# MIP solver selection (convert enum to int)
solver.updateSetting("MIP.Solver", "Dual", int(SHOTpy.MIPSolver.Gurobi))  # or Cplex, Cbc, Highs

# NLP solver selection
solver.updateSetting("FixedInteger.Solver", "Primal", int(SHOTpy.PrimalNLPSolver.Ipopt))  # or GAMS

print("Settings configured successfully!")

# Verify some settings
print(f"Time limit: {solver.getDoubleSetting('TimeLimit', 'Termination')} seconds")
print(f"Log level: {solver.getIntSetting('Console.LogLevel', 'Output')}")

## 9. Advanced Features {#advanced-features}

### Monomial and Signomial Terms

For geometric programming, you can use signomial terms:

In [ ]:
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Create a signomial element: x^2 * y^(-1)
element1 = SHOTpy.SignomialElement(x, 2.0)
element2 = SHOTpy.SignomialElement(y, -1.0)

# Create signomial term: 3.0 * x^2 * y^(-1) - coefficient first, then elements
signomial = SHOTpy.SignomialTerm(3.0, [element1, element2])

# Alternative: Create monomial using variable-power pairs directly
monomial = SHOTpy.SignomialTerm(2.5, [(x, 2.0), (y, 3.0)])

# Use in constraint
constraint = SHOTpy.NonlinearConstraint(0, "signomial_constraint", -SHOTpy.SHOT_DBL_MAX, 100.0)
constraint.add(signomial)
problem.addConstraint(constraint)

# Simple objective
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)

problem.finalize()
print("Problem with signomial terms created")
print(problem.toString())

### Examining Solution Details

In [ ]:
# Solve a simple problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

objective = SHOTpy.QuadraticObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.QuadraticTerm(1.0, x, x))
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))
problem.setObjective(objective)

constraint = SHOTpy.LinearConstraint(0, "sum", 3.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

# Get detailed results
print("Solution Details:")
sol = solver.getPrimalSolution()
print(f"  Primal objective: {sol.objValue:.6f}")
print(f"  Dual bound: {solver.getCurrentDualBound():.6f}")

solutions = solver.getPrimalSolutions()
print(f"  Number of solutions found: {len(solutions)}")

if len(solutions) > 0:
    print(f"  Best solution: x = {solutions[0].point[0]:.4f}, y = {solutions[0].point[1]:.4f}")

# Get problem name (if set)
print(f"  Problem name: {problem.name if hasattr(problem, 'name') else 'N/A'}")

### Checking Problem Properties

In [ ]:
# Create a problem with various features
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 5.0)
problem.addVariable(x)
problem.addVariable(y)

# Mix of linear, quadratic, and nonlinear terms
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(2.0, x))
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))
objective.add(SHOTpy.log(x))
problem.setObjective(objective)

constraint1 = SHOTpy.LinearConstraint(0, "linear", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint1.add(SHOTpy.LinearTerm(1.0, x))
problem.addConstraint(constraint1)

constraint2 = SHOTpy.NonlinearConstraint(1, "nonlinear", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint2.add(SHOTpy.exp(x))
problem.addConstraint(constraint2)

problem.finalize()

# Display problem structure with indicators
print("Problem Structure:")
print(problem.toString())
print("\nIndicators legend:")
print("  L = Linear terms")
print("  Q = Quadratic terms")
print("  S = Signomial terms (power functions)")
print("  M = Monomial terms")
print("  E = General nonlinear expressions")

## Summary

This tutorial covered:

1. **Basic setup** - Creating environments and problems
2. **Variables** - Real, integer, and binary variables
3. **Linear programming** - LP formulation and solving
4. **Mixed-integer programming** - MIP problems
5. **Quadratic programming** - QP and QCQP problems
6. **Nonlinear programming** - Using exp, log, sin, cos, and other functions
7. **File I/O** - Reading problems from OSiL and GAMS files
8. **Settings** - Configuring solver parameters
9. **Advanced features** - Signomial terms, solution details, and problem properties

### Additional Resources

- **SHOT Documentation**: https://shotsolver.dev
- **SHOT Repository**: https://github.com/coin-or/SHOT
- **Test Files**: See the `test/python/` directory for more examples

### Tips for Best Results

1. Always call `problem.finalize()` before creating a solver
2. Use appropriate variable bounds to help the solver
3. Configure MIP and NLP subsolvers based on problem characteristics
4. Adjust termination criteria (time limit, gap tolerances) as needed
5. Check solution status and bounds after solving